

# 04 Regression

The following notebook builds on previous notebook. after having observed how different metrics differ given our ai classification strategy, we want to perform linear regression in order to see how ai as a classifier might impact our metrics in question 

----

growth dynamics

    - revenue growth 

    - employment growth

    - innovation, R&D 

financial health

    - profitability 

    - returns (on assets, investments)



In [37]:
# this file is used to find the project root and set the working directory to it.
from pathlib import Path
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt




In [38]:
def find_project_root(marker=".project-root"):
    path = Path.cwd().resolve()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker}")

PROJECT_ROOT = find_project_root()
print(f"Project root found at: {PROJECT_ROOT}")

cwd = PROJECT_ROOT



Project root found at: /Users/ducjeremyvu/mime/sem-1/files/bank-fintech/essay


In [39]:
# data.columns.to_list()

In [40]:
data = pd.read_pickle(cwd / "data/babina_cleaned_enriched.pkl")

selection = [
'gvkey',
'year',
'aiempl',
'totalempl',
'aiemp_gt_0',
'aiempl_ratio',
'ai_intensity',
'n_years',
'first_year',
'last_year',
'market_cap',
'revenue_growth',
'log_revenue_growth',
'employment_growth',
'roa',
'ni_over_totalempl',
'log_totalempl',
'log_market_cap',
]


data[selection].describe()

,gvkey,year,aiempl,totalempl,aiempl_ratio,n_years,first_year,last_year,market_cap,revenue_growth,log_revenue_growth,employment_growth,roa,ni_over_totalempl,log_totalempl,log_market_cap
count,50622.000000,50622.000000,50622.000000,5.062200e+04,50622.000000,50622.000000,50622.000000,50622.000000,50622.000000,42769.000000,42769.000000,42769.000000,50622.000000,50622.000000,50622.000000,50622.000000
mean,76430.701948,2011.316740,8.094386,6.576617e+03,0.000918,9.987673,2006.465272,2016.091363,28919.327781,4.496306,-0.001482,0.074747,0.208662,6.778142,6.812129,9.656472
std,73493.815943,3.975979,97.511928,2.717678e+04,0.004329,4.193476,2.723738,3.187312,25594.846322,113.021840,1.401740,0.185891,11.860670,38.261310,1.986406,1.414921
min,1004.000000,2005.000000,0.000000,1.000000e+00,0.000000,1.000000,2005.000000,2005.000000,0.417556,-0.999923,-9.471966,-0.752991,-19.073949,-142.479392,0.000000,-0.873335
25%,13839.000000,2008.000000,0.000000,2.360000e+02,0.000000,7.000000,2005.000000,2015.000000,7834.478710,-0.501700,-0.696554,0.009921,0.009154,0.085365,5.463832,8.966290
50%,31208.500000,2011.000000,0.000000,8.920000e+02,0.000000,11.000000,2005.000000,2018.000000,21567.542398,-0.002463,-0.002466,0.046440,0.026875,0.648391,6.793466,9.978945
75%,148224.000000,2015.000000,1.000000,3.227000e+03,0.000348,14.000000,2007.000000,2018.000000,43950.966878,0.981806,0.684009,0.097720,0.055233,3.233194,8.079308,10.690830
max,328795.000000,2018.000000,6190.000000,1.552543e+06,0.400000,14.000000,2018.000000,2018.000000,115691.966503,16051.910568,9.683645,11.600000,2456.927679,2122.653912,14.255405,11.658686


We will first start regressing on growth rates. later on we will then explore other metrics 

In [ ]:
import statsmodels.formula.api as smf

from linearmodels.panel import PanelOLS

from linearmodels.panel import compare





# Revenue Growth
## Simple Linear regression 
### simple binary classifier

In [56]:
reg_data = data.dropna(subset=[
    # need to drop because during computation of growth rates, we get rows with NA for the first year of observation
    "revenue_growth",
    "aiempl_ratio",
    "log_totalempl",
    "log_market_cap",
    "roa",
    "gvkey",
    "year"
]).copy()


model_revgrowth_bin = smf.ols(
    "revenue_growth ~ aiemp_gt_0",
    data=reg_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gvkey"]}
)

print(model_revgrowth_bin.summary())

                            OLS Regression Results                            
Dep. Variable:         revenue_growth   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.3386
Date:                Sat, 27 Jun 2026   Prob (F-statistic):              0.561
Time:                        14:33:21   Log-Likelihood:            -2.6288e+05
No. Observations:               42769   AIC:                         5.258e+05
Df Residuals:                   42767   BIC:                         5.258e+05
Df Model:                           1                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              4.2222      0

### Intensity Classifier

In [43]:
model_revgrowth_ai_int = smf.ols(
    "revenue_growth ~ ai_intensity",
    data=reg_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gvkey"]}
)

print(model_revgrowth_ai_int.summary())

                            OLS Regression Results                            
Dep. Variable:         revenue_growth   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.4110
Date:                Sat, 27 Jun 2026   Prob (F-statistic):              0.745
Time:                        14:30:06   Log-Likelihood:            -2.6288e+05
No. Observations:               42769   AIC:                         5.258e+05
Df Residuals:                   42765   BIC:                         5.258e+05
Df Model:                           3                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  3

### Ai Empl

In [44]:
model_revgrowth_ai_ratio = smf.ols(
    "revenue_growth ~ aiempl_ratio",
    data=reg_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_data["gvkey"]}
)

print(model_revgrowth_ai_ratio.summary())

                            OLS Regression Results                            
Dep. Variable:         revenue_growth   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                  0.005938
Date:                Sat, 27 Jun 2026   Prob (F-statistic):              0.939
Time:                        14:30:06   Log-Likelihood:            -2.6288e+05
No. Observations:               42769   AIC:                         5.258e+05
Df Residuals:                   42767   BIC:                         5.258e+05
Df Model:                           1                                         
Covariance Type:              cluster                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        4.4905      0.550      8.160   

In [45]:
from statsmodels.iolib.summary2 import summary_col

summary_col(
    [model_revgrowth_bin, model_revgrowth_ai_int, model_revgrowth_ai_ratio],
    stars=True,
    model_names=["Simple Binary", "Intensity", "Ratio"],
    info_dict={
        "N": lambda x: f"{int(x.nobs)}",
        "R²": lambda x: f"{x.rsquared:.3f}"
    }
)

,Simple Binary,Intensity,Ratio
Intercept,4.2222***,3.5366***,4.4905***
,(0.5356),(0.8740),(0.5503)
aiemp_gt_0[T.True],0.7452,,
,(1.2807),,
ai_intensity[T.Low AI],,0.7534,
,,(1.2956),
ai_intensity[T.Mid AI],,2.5714,
,,(2.4960),
ai_intensity[T.No AI],,0.6855,
,,(1.0252),


## Adding for Controls and fixed effects

For the baseline model, we use the AI employment ratio and the binary AI employment indicator, while excluding the AI intensity classifier. Since the intensity classifier is derived directly from the AI employment ratio, including it would provide largely redundant information.

The theoretical justification is to run regression given the following model: 

$ y =  \beta_0 + \beta_1 aiempl\_ratio + \beta_C X_C + \epsilon $

or 

$ y =  \beta_0 + \beta_1 aiemp\_gt\_0 + \beta_C X_C + \epsilon $

with $X_C$ being the vector of control variables. We add controls for: 

- log(total_empl)
- log(market_cap)
- roa

and later models add for:  
- firm fixed effects
- year fixed effects 

In [63]:
from linearmodels.panel import PanelOLS

reg_fe = reg_data.set_index(["gvkey", "year"])


### Regression with Controls only
#### Binary Case

In [77]:
model_revgrowth_binary_control = PanelOLS.from_formula(
    """
    revenue_growth ~ aiemp_gt_0
                   + log_totalempl
                   + log_market_cap
                   + roa
    """,
    data=reg_fe
)

results_model_revgrowth_binary_control = model_revgrowth_binary_control.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_revgrowth_binary_control)

                          PanelOLS Estimation Summary                           
Dep. Variable:         revenue_growth   R-squared:                        0.0015
Estimator:                   PanelOLS   R-squared (Between):              0.0105
No. Observations:               42769   R-squared (Within):           -6.259e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):              0.0015
Time:                        14:40:40   Log-likelihood                -2.629e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      16.420
Entities:                        6441   P-value                           0.0000
Avg Obs:                       6.6401   Distribution:                 F(4,42765)
Min Obs:                       1.0000                                           
Max Obs:                       13.000   F-statistic (robust):             22.538
                            

#### Ratio case

In [78]:
model_revgrowth_ratio_control = PanelOLS.from_formula(
    """
    revenue_growth ~ aiempl_ratio
                   + log_totalempl
                   + log_market_cap
                   + roa
    """,
    data=reg_fe
)

results_model_revgrowth_ratio_control = model_revgrowth_ratio_control.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_revgrowth_ratio_control)



                          PanelOLS Estimation Summary                           
Dep. Variable:         revenue_growth   R-squared:                        0.0015
Estimator:                   PanelOLS   R-squared (Between):              0.0104
No. Observations:               42769   R-squared (Within):           -4.793e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):              0.0015
Time:                        14:40:41   Log-likelihood                -2.629e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      16.392
Entities:                        6441   P-value                           0.0000
Avg Obs:                       6.6401   Distribution:                 F(4,42765)
Min Obs:                       1.0000                                           
Max Obs:                       13.000   F-statistic (robust):             21.141
                            

### Adding Fixed Effects

#### Binary Case

In [82]:



model_revgrowth_binary_fe = PanelOLS.from_formula(
    """
    revenue_growth ~ aiemp_gt_0 
                   + log_totalempl 
                   + log_market_cap 
                   + roa
                   + EntityEffects
                   + TimeEffects
    """,
    data=reg_fe
)

results_model_revgrowth_binary_fe = model_revgrowth_binary_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_revgrowth_binary_fe)

                          PanelOLS Estimation Summary                           
Dep. Variable:         revenue_growth   R-squared:                     8.578e-05
Estimator:                   PanelOLS   R-squared (Between):             -0.2537
No. Observations:               42769   R-squared (Within):            2.219e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):             -0.0406
Time:                        14:42:12   Log-likelihood                -2.606e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.7788
Entities:                        6441   P-value                           0.5387
Avg Obs:                       6.6401   Distribution:                 F(4,36312)
Min Obs:                       1.0000                                           
Max Obs:                       13.000   F-statistic (robust):             0.6918
                            

#### Ratio Case

In [109]:

model_revgrowth_ratio_fe = PanelOLS.from_formula(
    """
    revenue_growth ~ aiempl_ratio 
                   + log_totalempl 
                   + log_market_cap 
                   + roa
                   + EntityEffects
                   + TimeEffects
    """,
    data=reg_fe
)

results_model_revgrowth_ratio_fe = model_revgrowth_ratio_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_revgrowth_ratio_fe)

                          PanelOLS Estimation Summary                           
Dep. Variable:         revenue_growth   R-squared:                       6.8e-05
Estimator:                   PanelOLS   R-squared (Between):             -0.2820
No. Observations:               42769   R-squared (Within):            1.633e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):             -0.0444
Time:                        15:32:01   Log-likelihood                -2.606e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.6173
Entities:                        6441   P-value                           0.6501
Avg Obs:                       6.6401   Distribution:                 F(4,36312)
Min Obs:                       1.0000                                           
Max Obs:                       13.000   F-statistic (robust):             0.8721
                            

In [83]:


comparison = compare({
    "Binary + Controls": results_model_revgrowth_binary_control,
    "Ratio + Controls": results_model_revgrowth_ratio_control,
    "Binary + FE": results_model_revgrowth_binary_fe,
    "Ratio + FE": results_model_revgrowth_ratio_fe,
})

print(comparison)

                                          Model Comparison                                         
                         Binary + Controls   Ratio + Controls        Binary + FE         Ratio + FE
---------------------------------------------------------------------------------------------------
Dep. Variable               revenue_growth     revenue_growth     revenue_growth     revenue_growth
Estimator                         PanelOLS           PanelOLS           PanelOLS           PanelOLS
No. Observations                     42769              42769              42769              42769
Cov. Est.                        Clustered          Clustered          Clustered          Clustered
R-squared                           0.0015             0.0015          8.578e-05            6.8e-05
R-Squared (Within)              -6.259e-05         -4.793e-05          2.219e-05          1.633e-05
R-Squared (Between)                 0.0105             0.0104            -0.2537            -0.2820


# Market Cap 

In [110]:
model_lmc_binary_control = PanelOLS.from_formula(
    """
    log_market_cap ~ aiemp_gt_0
                   + log_totalempl
                   + roa
    """,
    data=reg_fe
)

results_model_lmc_binary_control = model_lmc_binary_control.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_lmc_binary_control)

                          PanelOLS Estimation Summary                           
Dep. Variable:         log_market_cap   R-squared:                        0.9238
Estimator:                   PanelOLS   R-squared (Between):              0.9077
No. Observations:               42769   R-squared (Within):              -0.1538
Date:                Sat, Jun 27 2026   R-squared (Overall):              0.9238
Time:                        15:40:26   Log-likelihood                -1.031e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   1.727e+05
Entities:                        6441   P-value                           0.0000
Avg Obs:                       6.6401   Distribution:                 F(3,42766)
Min Obs:                       1.0000                                           
Max Obs:                       13.000   F-statistic (robust):          2.847e+04
                            

In [ ]:
model_lmc_ratio_control = PanelOLS.from_formula(
    """
    log_market_cap ~ aiempl_ratio
                   + log_totalempl
                   + roa
    """,
    data=reg_fe
)

results_model_lmc_ratio_control = model_lmc_ratio_control.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_lmc_ratio_control)

                          PanelOLS Estimation Summary                           
Dep. Variable:         log_market_cap   R-squared:                        0.9160
Estimator:                   PanelOLS   R-squared (Between):              0.8953
No. Observations:               42769   R-squared (Within):              -0.0525
Date:                Sat, Jun 27 2026   R-squared (Overall):              0.9160
Time:                        15:40:52   Log-likelihood                -1.052e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   1.554e+05
Entities:                        6441   P-value                           0.0000
Avg Obs:                       6.6401   Distribution:                 F(3,42766)
Min Obs:                       1.0000                                           
Max Obs:                       13.000   F-statistic (robust):          1.945e+04
                            

In [112]:

model_lmc_binary_fe = PanelOLS.from_formula(
    """
    log_market_cap ~ aiemp_gt_0 
                   + log_totalempl  
                   + roa
                   + EntityEffects
                   + TimeEffects
    """,
    data=reg_fe
)

results_model_lmc_binary_fe = model_lmc_binary_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_lmc_binary_fe)

                          PanelOLS Estimation Summary                           
Dep. Variable:         log_market_cap   R-squared:                     5.078e-05
Estimator:                   PanelOLS   R-squared (Between):              0.0547
No. Observations:               42769   R-squared (Within):            6.567e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):              0.0611
Time:                        15:42:52   Log-likelihood                -7.207e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.6147
Entities:                        6441   P-value                           0.6054
Avg Obs:                       6.6401   Distribution:                 F(3,36313)
Min Obs:                       1.0000                                           
Max Obs:                       13.000   F-statistic (robust):             1.3130
                            

In [113]:

model_lmc_ratio_fe = PanelOLS.from_formula(
    """
    log_market_cap ~ aiempl_ratio 
                   + log_totalempl  
                   + roa
                   + EntityEffects
                   + TimeEffects
    """,
    data=reg_fe
)

results_model_lmc_ratio_fe = model_lmc_ratio_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_lmc_ratio_fe)

                          PanelOLS Estimation Summary                           
Dep. Variable:         log_market_cap   R-squared:                     4.114e-05
Estimator:                   PanelOLS   R-squared (Between):              0.0471
No. Observations:               42769   R-squared (Within):            5.593e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):              0.0527
Time:                        15:42:57   Log-likelihood                -7.207e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.4980
Entities:                        6441   P-value                           0.6836
Avg Obs:                       6.6401   Distribution:                 F(3,36313)
Min Obs:                       1.0000                                           
Max Obs:                       13.000   F-statistic (robust):             1.3008
                            

In [114]:


comparison = compare({
    "Binary + Controls": results_model_lmc_binary_control,
    "Ratio + Controls": results_model_lmc_ratio_control,
    "Binary + FE": results_model_lmc_binary_fe,
    "Ratio + FE": results_model_lmc_ratio_fe,
})

print(comparison)

                                          Model Comparison                                         
                         Binary + Controls   Ratio + Controls        Binary + FE         Ratio + FE
---------------------------------------------------------------------------------------------------
Dep. Variable               log_market_cap     log_market_cap     log_market_cap     log_market_cap
Estimator                         PanelOLS           PanelOLS           PanelOLS           PanelOLS
No. Observations                     42769              42769              42769              42769
Cov. Est.                        Clustered          Clustered          Clustered          Clustered
R-squared                           0.9238             0.9160          5.078e-05          4.114e-05
R-Squared (Within)                 -0.1538            -0.0525          6.567e-05          5.593e-05
R-Squared (Between)                 0.9077             0.8953             0.0547             0.0471


# Patents



In [ ]:
# using full data, because not dependent on growth rates 



panel_ind = data.set_index(["gvkey", "year"])

panel = panel_ind.dropna(subset=[
    "log_productpatent1_fyear_weightvalue",  # add this
    "aiempl_ratio",
    "log_totalempl",
    "log_market_cap",
    "roa",
]).copy()


model_patent_binary_control = PanelOLS.from_formula(
    """
    log_productpatent1_fyear_weightvalue ~ aiemp_gt_0
                   + log_totalempl
                   + log_market_cap
                   + roa
    """,
    data=panel
)

results_model_patent_binary_control = model_patent_binary_control.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_patent_binary_control)

                                   PanelOLS Estimation Summary                                    
Dep. Variable:     log_productpatent1_fyear_weightvalue   R-squared:                        0.5973
Estimator:                                     PanelOLS   R-squared (Between):              0.4664
No. Observations:                                 16335   R-squared (Within):              -0.0832
Date:                                  Sat, Jun 27 2026   R-squared (Overall):              0.5973
Time:                                          15:25:55   Log-likelihood                  -3.6e+04
Cov. Estimator:                               Clustered                                           
                                                          F-statistic:                      6055.6
Entities:                                          2793   P-value                           0.0000
Avg Obs:                                         5.8485   Distribution:                 F(4,16331)
Min Obs:  

In [94]:

model_patent_ratio_control = PanelOLS.from_formula(
    """
    log_productpatent1_fyear_weightvalue ~ aiempl_ratio
                   + log_totalempl
                   + log_market_cap
                   + roa
    """,
    data=panel
)

results_model_patent_ratio_control = model_patent_ratio_control.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_patent_ratio_control)

                                   PanelOLS Estimation Summary                                    
Dep. Variable:     log_productpatent1_fyear_weightvalue   R-squared:                        0.5971
Estimator:                                     PanelOLS   R-squared (Between):              0.4598
No. Observations:                                 16335   R-squared (Within):              -0.0914
Date:                                  Sat, Jun 27 2026   R-squared (Overall):              0.5971
Time:                                          15:14:26   Log-likelihood                  -3.6e+04
Cov. Estimator:                               Clustered                                           
                                                          F-statistic:                      6050.6
Entities:                                          2793   P-value                           0.0000
Avg Obs:                                         5.8485   Distribution:                 F(4,16331)
Min Obs:  

In [98]:
model_patent_bin_fe = PanelOLS.from_formula(
    """
    log_productpatent1_fyear_weightvalue ~ aiemp_gt_0
                   + log_totalempl 
                   + log_market_cap 
                   + roa
                   + EntityEffects
                   + TimeEffects
    """,
    data=panel
)

results_model_patent_bin_fe = model_patent_bin_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_patent_bin_fe)

                                   PanelOLS Estimation Summary                                    
Dep. Variable:     log_productpatent1_fyear_weightvalue   R-squared:                        0.0185
Estimator:                                     PanelOLS   R-squared (Between):             -1.7437
No. Observations:                                 16335   R-squared (Within):              -0.0302
Date:                                  Sat, Jun 27 2026   R-squared (Overall):             -0.3213
Time:                                          15:25:07   Log-likelihood                -2.224e+04
Cov. Estimator:                               Clustered                                           
                                                          F-statistic:                      63.787
Entities:                                          2793   P-value                           0.0000
Avg Obs:                                         5.8485   Distribution:                 F(4,13525)
Min Obs:  

In [99]:
model_patent_ratio_fe = PanelOLS.from_formula(
    """
    log_productpatent1_fyear_weightvalue ~ aiempl_ratio 
                   + log_totalempl 
                   + log_market_cap 
                   + roa
                   + EntityEffects
                   + TimeEffects
    """,
    data=panel
)

results_model_patent_ratio_fe = model_patent_ratio_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_patent_ratio_fe)

                                   PanelOLS Estimation Summary                                    
Dep. Variable:     log_productpatent1_fyear_weightvalue   R-squared:                        0.0194
Estimator:                                     PanelOLS   R-squared (Between):             -1.5229
No. Observations:                                 16335   R-squared (Within):              -0.0338
Date:                                  Sat, Jun 27 2026   R-squared (Overall):             -0.2128
Time:                                          15:25:09   Log-likelihood                -2.223e+04
Cov. Estimator:                               Clustered                                           
                                                          F-statistic:                      66.932
Entities:                                          2793   P-value                           0.0000
Avg Obs:                                         5.8485   Distribution:                 F(4,13525)
Min Obs:  

In [101]:


comparison = compare({
    "Binary + Controls": results_model_patent_binary_control,
    "Ratio + Controls": results_model_patent_ratio_control,
    "Binary + FE": results_model_patent_bin_fe,
    "Ratio + FE": results_model_patent_ratio_fe,
})

print(comparison)

                                                                                      Model Comparison                                                                                     
                                               Binary + Controls                         Ratio + Controls                              Binary + FE                               Ratio + FE
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable               log_productpatent1_fyear_weightvalue     log_productpatent1_fyear_weightvalue     log_productpatent1_fyear_weightvalue     log_productpatent1_fyear_weightvalue
Estimator                                               PanelOLS                                 PanelOLS                                 PanelOLS                                 PanelOLS
No. Observations                                           1

# ROA

In [95]:
model_roa_binary_control = PanelOLS.from_formula(
    """
    roa ~ aiemp_gt_0
                   + log_totalempl
                   + log_market_cap
                   
    """,
    data=panel
)

results_model_roa_binary_control = model_roa_binary_control.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_roa_binary_control)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    roa   R-squared:                        0.0127
Estimator:                   PanelOLS   R-squared (Between):              0.0446
No. Observations:               16335   R-squared (Within):           -5.259e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):              0.0127
Time:                        15:22:47   Log-likelihood                -2.173e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      70.243
Entities:                        2793   P-value                           0.0000
Avg Obs:                       5.8485   Distribution:                 F(3,16332)
Min Obs:                       1.0000                                           
Max Obs:                       14.000   F-statistic (robust):             68.502
                            

In [96]:
model_roa_ratio_control = PanelOLS.from_formula(
    """
    roa ~ aiempl_ratio
                   + log_totalempl
                   + log_market_cap
                   
    """,
    data=panel
)

results_model_roa_ratio_control = model_roa_ratio_control.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_roa_ratio_control)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    roa   R-squared:                        0.0128
Estimator:                   PanelOLS   R-squared (Between):              0.0446
No. Observations:               16335   R-squared (Within):           -3.838e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):              0.0128
Time:                        15:23:14   Log-likelihood                -2.173e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      70.332
Entities:                        2793   P-value                           0.0000
Avg Obs:                       5.8485   Distribution:                 F(3,16332)
Min Obs:                       1.0000                                           
Max Obs:                       14.000   F-statistic (robust):             80.440
                            

In [106]:
model_roa_bin_fe = PanelOLS.from_formula(
    """
    roa ~ aiemp_gt_0
                   + log_totalempl 
                   + log_market_cap 
                   + EntityEffects
                   + TimeEffects
    """,
    data=panel
)

results_model_roa_bin_fe = model_roa_bin_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_roa_bin_fe)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    roa   R-squared:                     4.752e-05
Estimator:                   PanelOLS   R-squared (Between):              0.0201
No. Observations:               16335   R-squared (Within):            5.634e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):              0.0025
Time:                        15:29:19   Log-likelihood                -2.031e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.2143
Entities:                        2793   P-value                           0.8866
Avg Obs:                       5.8485   Distribution:                 F(3,13526)
Min Obs:                       1.0000                                           
Max Obs:                       14.000   F-statistic (robust):             0.1981
                            

In [107]:
model_roa_ratio_fe = PanelOLS.from_formula(
    """
    roa ~ aiempl_ratio 
                   + log_totalempl 
                   + log_market_cap 
                   + EntityEffects
                   + TimeEffects
    """,
    data=panel
)

results_model_roa_ratio_fe = model_roa_ratio_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_model_roa_ratio_fe)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    roa   R-squared:                     9.168e-05
Estimator:                   PanelOLS   R-squared (Between):             -0.0199
No. Observations:               16335   R-squared (Within):            9.692e-05
Date:                Sat, Jun 27 2026   R-squared (Overall):             -0.0122
Time:                        15:29:20   Log-likelihood                -2.031e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.4134
Entities:                        2793   P-value                           0.7434
Avg Obs:                       5.8485   Distribution:                 F(3,13526)
Min Obs:                       1.0000                                           
Max Obs:                       14.000   F-statistic (robust):             0.5380
                            

In [108]:


comparison = compare({
    "Binary + Controls": results_model_roa_binary_control,
    "Ratio + Controls": results_model_roa_ratio_control,
    "Binary + FE": results_model_roa_bin_fe,
    "Ratio + FE": results_model_roa_ratio_fe,
})

print(comparison)

                                   Model Comparison                                   
                        Binary + Controls Ratio + Controls   Binary + FE    Ratio + FE
--------------------------------------------------------------------------------------
Dep. Variable                         roa              roa           roa           roa
Estimator                        PanelOLS         PanelOLS      PanelOLS      PanelOLS
No. Observations                    16335            16335         16335         16335
Cov. Est.                       Clustered        Clustered     Clustered     Clustered
R-squared                          0.0127           0.0128     4.752e-05     9.168e-05
R-Squared (Within)             -5.259e-05       -3.838e-05     5.634e-05     9.692e-05
R-Squared (Between)                0.0446           0.0446        0.0201       -0.0199
R-Squared (Overall)                0.0127           0.0128        0.0025       -0.0122
F-statistic                        70.243  